# Preserving concurrency in the distribution representation

The `distribution` representation reproduces each attribute's **duration curve**
— the right set of values and how often each occurs — but it still has to decide
*how to place those values in time* within each typical period. The
`concurrency` option of `Distribution` controls that.

Every strategy preserves each attribute's duration curve **identically**; they
differ only in how well they keep the *joint* structure across attributes (which
values co-occur — e.g. is solar low exactly when demand is high?).

- `"independent"` *(default)* — orders each attribute on its own; joint structure lost.
- `"medoid"` — orders every attribute by one real period's ranks; keeps the joint structure.
- `"reference"` — broadcasts one attribute's ordering (needs `reference_attribute`).
- `"consensus"` / `"assignment"` — one shared ordering for all attributes.

This notebook shows how to switch it on and how to confirm it helped, on a small
two-attribute example (solar `GHI` vs `Load`).

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc[
    "2010-01-01":"2010-01-21", ["GHI", "Load"]
]  # three weeks, two attributes
data.head()

## Switch it on

`concurrency` lives on the `Distribution` representation. Everything else about
the aggregation stays the same.

In [ ]:
def aggregate(concurrency):
    return tsam.aggregate(
        data,
        n_clusters=3,
        period_duration="1D",
        cluster=ClusterConfig(
            method="hierarchical",
            representation=Distribution(concurrency=concurrency),
        ),
        preserve_column_means=False,
    )


independent = aggregate("independent")
medoid = aggregate("medoid")

## Confirm it helped

Two metrics tell the whole story:

- `result.accuracy.weighted_rmse_duration` — the **marginal** error (per-attribute
  distribution). Identical for every strategy.
- `result.concurrency` — the **joint** error (Frobenius norm of the difference
  between the original and reconstructed correlation matrices). Lower is better.

In [ ]:
pd.DataFrame(
    {
        name: {
            "rmse_duration": result.accuracy.weighted_rmse_duration,
            "correlation_error": result.concurrency.correlation_error,
        }
        for name, result in {"independent": independent, "medoid": medoid}.items()
    }
).T.round(4)

Same `rmse_duration` (the duration curves are untouched), but `medoid` cuts the
`correlation_error` by more than an order of magnitude — it keeps the `GHI`/`Load` co-occurrence that
`independent` discards, for free.

## See the decision

Each strategy *decides* where to place each attribute's values in time.
`result.plot.cluster_members` shows every cluster's real member days (faint) with
the chosen representative highlighted — that representative **is** the decision.
Clustering is identical, so both plots show the *same* clusters. Under `medoid`
the representative follows one real member's joint shape; under `independent`
each attribute is placed on its own, so the representative need not resemble any
real day.

In [ ]:
independent.plot.cluster_members(title="concurrency='independent'")

In [ ]:
medoid.plot.cluster_members(title="concurrency='medoid'")

## When to use which

- **`independent`** *(default)* — when only each attribute's own distribution
  matters. Best marginal fit, no cross-attribute structure.
- **`medoid`** — the go-to when co-occurrence between attributes matters
  (solar vs demand, wind vs demand, …). Best joint structure at no marginal cost.
- **`reference`** — when one attribute's timing should anchor the rest; set
  `reference_attribute="…"`.
- **`consensus` / `assignment`** — a single shared time axis for all attributes.

`concurrency` only applies to `scope="local"` (the default) distribution
representations.